# 00 · Baseline — Whisper transcript → sentence embedding → Ridge

**Goal:** the fastest honest path to a valid `submission.csv` (target: under an hour on a Kaggle GPU).

| step | what happens | cached to |
|---|---|---|
| 1 | find CSVs + audio, detect columns | – |
| 2 | load audio → mono 16 kHz → trim edge silence → normalise loudness | `cache/audio16k/` |
| 3 | local Whisper transcription (text **and** timestamps) | `cache/segments.jsonl`, `cache/transcripts.csv` |
| 4 | one sentence encoder embeds each transcript | `cache/baseline_emb_<hash>.npy` |
| 5 | Ridge, 5-fold stratified CV (RMSE + Pearson on every fold), constant baseline printed | – |
| 6 | fit on all training data, predict test, write + validate `submission.csv` | `OUTPUT_DIR/submission.csv` |

Whisper runs **once**: `01_extract_features.ipynb` and `final.ipynb` reuse the cache written here.

> ⚠️ **Untested code.** Written without access to the data, a GPU, or the internet. The pure-Python/sklearn parts were smoke-tested on synthetic data; the Whisper / encoder / Kaggle-path parts were not run. Expect to fix small things (see README "What is unverified").

In [ ]:
# Kaggle images already ship torch / transformers / librosa / lightgbm. Missing pieces (needs Internet ON):
!pip install -q faster-whisper sentence-transformers language-tool-python
# With Internet OFF: attach the wheels as a Kaggle Dataset and run:  !pip install --no-index --find-links=/kaggle/input/<wheels> faster-whisper

In [ ]:
# =====================  CONFIG  -- the ONLY cell you should need to edit  =====================
import sys
from pathlib import Path

CONFIG = dict(
    # ---- where the data is (Kaggle mounts it read-only; nothing is uploaded anywhere) ----
    DATA_DIR        = "/kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final",
    TRAIN_CSV       = None,    # None -> auto-discover train.csv under DATA_DIR
    TEST_CSV        = None,
    SAMPLE_SUB_CSV  = None,
    TRAIN_AUDIO_DIR = None,    # None -> auto-discover (folder with 'train' in its name that holds audio)
    TEST_AUDIO_DIR  = None,
    ID_COL          = None,    # None -> auto-detect the file-name column
    LABEL_COL       = None,    # None -> auto-detect the numeric label column
    # ---- where caches / results go ----
    CACHE_DIR       = "/kaggle/working/cache",
    OUTPUT_DIR      = "/kaggle/working",
    # ---- models: hub name (Internet ON)  OR  local / Kaggle-Models path (Internet OFF, set OFFLINE=True) ----
    OFFLINE         = False,
    MODEL_CACHE_DIR = None,
    WHISPER_MODEL   = "small",      # faster-whisper size, or path to a CTranslate2 Whisper folder
    BASELINE_TEXT_ENCODER = "sentence-transformers/all-MiniLM-L6-v2",
    TEXT_ENCODER    = "sentence-transformers/all-mpnet-base-v2",
    HF_ENCODER      = "roberta-base",
    LM_NAME         = "gpt2",
    AUDIO_ENCODER   = "microsoft/wavlm-base-plus",
    LLM_NAME        = "Qwen/Qwen2.5-1.5B-Instruct",
    # ---- switches ----
    RUN_LANGUAGETOOL = True,        # auto-skipped if Java / download is unavailable
    RUN_LLM_RUBRIC   = False,       # optional zero-shot rubric feature
    FORCE_RECOMPUTE  = False,       # True -> ignore cached feature files
    # ---- reproducibility / validation ----
    SEED = 42, CV_SPLITS = 5, CV_REPEATS = 3,
)

def _find_repo():
    cands = [Path.cwd(), Path.cwd().parent, Path("/kaggle/working/shl-grammar-scoring")]
    for pat in ("*", "*/*", "*/*/*", "*/*/*/*"):            # Kaggle nests datasets 1-4 levels deep
        cands += list(Path("/kaggle/input").glob(pat))
    for c in cands:
        if (c / "src" / "config.py").exists():
            return c
    raise FileNotFoundError("Cannot find the repo's src/ folder. Add the repo as a Kaggle Dataset (or clone it) and set REPO_DIR by hand.")

REPO_DIR = _find_repo()
sys.path.insert(0, str(REPO_DIR))
from src.config import Config
cfg = Config(**CONFIG).apply_env().seed_everything()      # env vars FIRST, before torch/transformers are imported
print("repo:", REPO_DIR)
print(cfg.summary())

## 1 · Load the competition files

In [ ]:
import numpy as np, pandas as pd
from IPython.display import display
from src.data import load_competition

data = load_competition(cfg)
meta = pd.concat([data["train"], data["test"]], ignore_index=True)   # train rows first, then test rows
print(meta.groupby("split").size().to_string())
display(meta.head(3))

## 2 · Preprocess audio
`librosa.load(..., sr=16000, mono=True)` resamples and mixes down; leading/trailing silence is trimmed (internal pauses are **kept** – they carry fluency information); loudness is normalised to a fixed RMS. Results are cached as 16-bit wav, so re-runs are instant.

In [ ]:
from src.audio import preprocess_all
meta = preprocess_all(meta, cfg)
display(meta[["split", "stem", "pre_ok", "cache_path"]].head(3))

## 3 · Transcribe locally with Whisper (cached, resumable)
Uses `faster-whisper` on the GPU. Text **and** segment/word timestamps are stored now so that `01_extract_features` never has to run Whisper again.  
Offline? Put a CTranslate2 Whisper model folder (e.g. from a Kaggle Model) in `CONFIG["WHISPER_MODEL"]` and set `OFFLINE=True`.

In [ ]:
from src.transcribe import transcribe_all, load_transcripts
from src.utils import free_gpu

records = transcribe_all(meta, cfg)       # appends to cache/segments.jsonl, writes cache/transcripts.csv
free_gpu()
tx = load_transcripts(cfg)
meta = meta.merge(tx[["split", "stem", "text"]], on=["split", "stem"], how="left")
meta["text"] = meta["text"].fillna("")
print("empty transcripts:", int((meta.text.str.len() == 0).sum()))
# (transcript text is deliberately NOT printed: it is derived from the competition data and must not be published)
print(meta.assign(n_words=meta.text.str.split().str.len()).groupby("split").n_words.describe().round(1).to_string())

## 4 · Embed each transcript (one sentence encoder)
`all-MiniLM-L6-v2` (small and fast). The cache filename contains a hash of the file list, so a changed dataset can never silently reuse stale embeddings.

In [ ]:
from src.embeddings import embed_sentence_transformer
from src.utils import keys_hash

emb_path = cfg.cache_dir / f"baseline_emb_{keys_hash(meta)}.npy"
if emb_path.exists() and not cfg.FORCE_RECOMPUTE:
    E = np.load(emb_path)
else:
    E = embed_sentence_transformer(meta["text"].tolist(), cfg.BASELINE_TEXT_ENCODER, cfg)
    np.save(emb_path, E)
assert E.shape[0] == len(meta)
free_gpu()
print("embedding matrix:", E.shape)

## 5 · Ridge with 5-fold stratified CV
* Folds are stratified on the **rounded** label (rare classes merged), seed fixed.  
* Imputer, scaler and the ridge penalty are fit **inside** each training fold (they live in one sklearn `Pipeline`) → no leakage.  
* Predictions are clipped to [0, 5].  
* **Constant baseline** = always predict the training-fold mean. Its RMSE ≈ the label standard deviation; a real model must beat it.

In [ ]:
from src.models import stratify_labels, run_cv, make_constant, make_ridge
from src.metrics import rmse, pearson, clip_scores

tr = (meta.split == "train").to_numpy(); te = ~tr
y = meta.loc[tr, "label"].to_numpy(float)
Xtr, Xte = E[tr], E[te]
strat = stratify_labels(y, cfg.CV_SPLITS)

print(f"Label std (RMSE of 'always predict the mean'): {y.std():.4f}")
const = run_cv(make_constant, Xtr, y, strat, [cfg.SEED], cfg.CV_SPLITS, "constant_mean", verbose=False)
print(f"Constant-mean baseline, CV RMSE: {const['summary']['rmse_overall']:.4f}  (Pearson undefined: constant prediction)\n")

base = run_cv(make_ridge, Xtr, y, strat, [cfg.SEED], cfg.CV_SPLITS, "ridge_minilm")
s = base["summary"]
results_log = pd.DataFrame([dict(experiment="baseline: MiniLM + Ridge", cv_rmse=s["rmse_overall"], cv_pearson=s["pearson_overall"])])
display(results_log.round(4))

## 6 · Fit on all training data → predict test → write and validate `submission.csv`

In [ ]:
from src.submission import write_submission

model = make_ridge().fit(Xtr, y)
train_pred = clip_scores(model.predict(Xtr))
test_pred = clip_scores(model.predict(Xte))

print("=" * 72)
print(f"TRAINING RMSE (baseline model, fit and scored on the full training set): {rmse(y, train_pred):.4f}")
print(f"CV RMSE     : {s['rmse_overall']:.4f}")
print(f"CV Pearson  : {s['pearson_overall']:.4f}")
print(f"Constant baseline RMSE (label std): {y.std():.4f}")
print("=" * 72)
print("(training RMSE is optimistic: the model has seen these labels. CV RMSE is the honest number.)")

sub = write_submission(test_pred, meta[te].reset_index(drop=True), data["test_csv"], str(cfg.output_dir / "submission.csv"))
display(sub.head())
print(sub.iloc[:, -1].describe().round(3).to_string())

**Next:** run `01_extract_features.ipynb` (it reuses the Whisper cache) and then `final.ipynb`.  
Keep `cache/` private: it is derived from the competition data and must not be published.